# 📰 Fake News Detector
### ML Project — Logistic Regression + BiLSTM with NLP
This notebook trains and evaluates two models to classify news articles as **Real** or **Fake**.

## Step 1: Clone the GitHub Repo

In [ ]:
!git clone https://github.com/shrxyash/fake-news-detector.git
%cd fake-news-detector

## Step 2: Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn nltk torch seaborn matplotlib shap
import nltk
nltk.download('stopwords')
print('All dependencies installed!')

## Step 3: Load and Preprocess Data

In [ ]:
import sys
sys.path.insert(0, '/content/fake-news-detector')

from src.preprocess import load_train, get_splits
import pandas as pd

df = load_train()
print('\n--- Dataset Overview ---')
print(df[['title', 'label']].head())
print(f'\nTotal articles: {len(df)}')
print(f'Fake: {df["label"].sum()} | Real: {(df["label"]==0).sum()}')

## Step 4: Label Distribution

In [ ]:
import matplotlib.pyplot as plt

counts = df['label'].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Real (0)', 'Fake (1)'], [counts[0], counts[1]], color=['steelblue', 'tomato'])
ax.set_title('Label Distribution')
ax.set_ylabel('Count')
for i, v in enumerate([counts[0], counts[1]]):
    ax.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 5: Train Logistic Regression Model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import numpy as np

X_train, X_val, y_train, y_val = get_splits()

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', class_weight='balanced', n_jobs=-1))
])

print('Training Logistic Regression...')
pipeline.fit(X_train, y_train)

val_pred  = pipeline.predict(X_val)
val_proba = pipeline.predict_proba(X_val)[:, 1]

print('\n--- Classification Report ---')
print(classification_report(y_val, val_pred, target_names=['Real', 'Fake']))
print(f'ROC-AUC Score: {roc_auc_score(y_val, val_proba):.4f}')

## Step 6: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_val, val_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

## Step 7: ROC Curve

In [ ]:
from sklearn.metrics import RocCurveDisplay

auc = roc_auc_score(y_val, val_proba)
fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_val, val_proba, ax=ax, name=f'LR (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_title('Logistic Regression — ROC Curve')
plt.tight_layout()
plt.show()

## Step 8: Top Predictive Words (TF-IDF Features)

In [ ]:
tfidf     = pipeline.named_steps['tfidf']
clf       = pipeline.named_steps['clf']
features  = tfidf.get_feature_names_out()
coefs     = clf.coef_[0]

top_fake = np.argsort(coefs)[-15:][::-1]
top_real = np.argsort(coefs)[:15]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh([features[i] for i in top_fake], [coefs[i] for i in top_fake], color='tomato')
axes[0].set_title('Top Words → FAKE News')
axes[0].invert_yaxis()

axes[1].barh([features[i] for i in top_real], [coefs[i] for i in top_real], color='steelblue')
axes[1].set_title('Top Words → REAL News')
axes[1].invert_yaxis()

plt.suptitle('Most Predictive Words by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 9: Train BiLSTM Model

In [ ]:
import sys
sys.path.insert(0, '/content/fake-news-detector')
from src.models.lstm_model import Vocabulary, NewsDataset, BiLSTM, run_epoch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

vocab = Vocabulary().build(X_train)
train_dl = DataLoader(NewsDataset(X_train, y_train, vocab), batch_size=64, shuffle=True)
val_dl   = DataLoader(NewsDataset(X_val,   y_val,   vocab), batch_size=64)

model     = BiLSTM(len(vocab)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
criterion = nn.BCEWithLogitsLoss()

history = []
print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>9}  {'Val Acc':>8}")
print('─' * 55)

for epoch in range(1, 6):  # 5 epochs for demo speed
    tr_loss, tr_acc = run_epoch(model, train_dl, optimizer, criterion, train=True)
    vl_loss, vl_acc = run_epoch(model, val_dl,   optimizer, criterion, train=False)
    scheduler.step(vl_loss)
    history.append((epoch, tr_loss, tr_acc, vl_loss, vl_acc))
    print(f'{epoch:>5}  {tr_loss:>10.4f}  {tr_acc:>9.4f}  {vl_loss:>9.4f}  {vl_acc:>8.4f}')

## Step 10: BiLSTM Training History

In [ ]:
epochs   = [h[0] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs, [h[1] for h in history], label='Train')
axes[0].plot(epochs, [h[3] for h in history], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs, [h[2] for h in history], label='Train')
axes[1].plot(epochs, [h[4] for h in history], label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

fig.suptitle('BiLSTM Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 11: Model Comparison Summary

In [ ]:
from sklearn.metrics import f1_score

# LR results
lr_auc = roc_auc_score(y_val, val_proba)
lr_f1  = f1_score(y_val, val_pred, average='macro')
lr_acc = (val_pred == y_val).mean()

# LSTM results
model.eval()
all_proba = []
with torch.no_grad():
    for x, _ in val_dl:
        all_proba.extend(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
lstm_proba = np.array(all_proba)
lstm_pred  = (lstm_proba >= 0.5).astype(int)
lstm_auc   = roc_auc_score(y_val, lstm_proba)
lstm_f1    = f1_score(y_val, lstm_pred, average='macro')
lstm_acc   = (lstm_pred == y_val).mean()

print('┌─────────────────────┬──────────┬──────────┬──────────┐')
print('│ Model               │ Accuracy │ ROC-AUC  │ F1 Macro │')
print('├─────────────────────┼──────────┼──────────┼──────────┤')
print(f'│ Logistic Regression │ {lr_acc:.4f}   │ {lr_auc:.4f}   │ {lr_f1:.4f}   │')
print(f'│ BiLSTM              │ {lstm_acc:.4f}   │ {lstm_auc:.4f}   │ {lstm_f1:.4f}   │')
print('└─────────────────────┴──────────┴──────────┴──────────┘')

# ROC Comparison plot
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_val, val_proba,   ax=ax, name=f'LR (AUC={lr_auc:.3f})')
RocCurveDisplay.from_predictions(y_val, lstm_proba,  ax=ax, name=f'BiLSTM (AUC={lstm_auc:.3f})')
ax.plot([0,1],[0,1],'k--',lw=0.8)
ax.set_title('Model Comparison — ROC Curves')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Step 12: Live Prediction Demo

In [ ]:
from src.preprocess import clean_text

test_articles = [
    "Scientists discover new vaccine that eliminates cancer completely in clinical trials",
    "Government secretly puts microchips in water supply to control the population",
    "Stock markets rise as inflation data comes in lower than expected",
    "Aliens have landed in New York and the president is hiding it from the public"
]

print(f'{'Article':<65} {'Prediction':<12} {'Confidence'}')
print('─' * 95)
for article in test_articles:
    cleaned = clean_text(article)
    proba   = pipeline.predict_proba([cleaned])[0][1]
    label   = 'FAKE 🚨' if proba >= 0.5 else 'REAL ✅'
    conf    = proba if proba >= 0.5 else 1 - proba
    print(f'{article[:65]:<65} {label:<12} {conf:.1%}')